# 03 - Earnings Call Transcript NLP Analysis
## CAVA Group Investment Analysis

**Objective:** Analyze CAVA's earnings call transcripts to:
1. Track management tone changes over time using VADER sentiment
2. Monitor keyword frequency for key investment themes
3. Validate modeling conclusions with qualitative evidence
4. Identify leading language signals that precede SSSG changes

**Data:** 11 earnings call transcripts (Q2 2023 – Q4 2025) from Capital IQ

**Approach:**
- Extract text from PDFs using pdfplumber
- Separate prepared remarks from Q&A section
- Apply VADER sentiment scoring
- Track keyword frequency across categories:
  - **Growth signals:** traffic, momentum, strong, exceed, accelerate
  - **Caution signals:** uncertain, pressure, challenging, macro, cautious
  - **SSSG-specific:** same-restaurant, comparable, comp
  - **Digital:** digital, online, app, delivery

In [1]:
import os
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import pdfplumber
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'axes.grid': False,
    'legend.fontsize': 11,
    'axes.labelsize': 12,
    'axes.titlesize': 13,
})

BASE_DIR = '/Users/zhangsu/Cava_Investment_Analysis'
TRANSCRIPT_DIR = f'{BASE_DIR}/data/earnings_transcripts'
analyzer = SentimentIntensityAnalyzer()

print('Libraries loaded.')

Libraries loaded.


---
## Part 1: Load & Extract Transcripts

In [2]:
# ── Transcript filename to quarter mapping ─────────────────────────────────────
transcript_map = {
    'CAVA Group Inc._Earnings Call_2023-08-15_English.pdf': 'Q2 2023',
    'CAVA Group Inc._Earnings Call_2023-11-07_English.pdf': 'Q3 2023',
    'CAVA Group Inc._Earnings Call_2024-02-27_English.pdf': 'Q4 2023',
    'CAVA Group Inc._Earnings Call_2024-05-28_English.pdf': 'Q1 2024',
    'CAVA Group Inc._Earnings Call_2024-08-22_English.pdf': 'Q2 2024',
    'CAVA Group Inc._Earnings Call_2024-11-12_English.pdf': 'Q3 2024',
    'CAVA Group Inc._Earnings Call_2025-02-25_English.pdf': 'Q4 2024',
    'CAVA Group Inc._Earnings Call_2025-05-15_English.pdf': 'Q1 2025',
    'CAVA Group Inc._Earnings Call_2025-08-12_English.pdf': 'Q2 2025',
    'CAVA Group Inc._Earnings Call_2025-11-04_English.pdf': 'Q3 2025',
    'CAVA Group Inc._Earnings Call_2026-02-24_English.pdf': 'Q4 2025',
}

# ── Extract text from PDFs ─────────────────────────────────────────────────────
def extract_transcript_text(pdf_path):
    """Extract full text from earnings call PDF using pdfplumber"""
    text = ''
    try:
        with pdfplumber.open(pdf_path) as pdf:
            for page in pdf.pages:
                page_text = page.extract_text()
                if page_text:
                    text += page_text + '\n'
    except Exception as e:
        print(f'Error extracting {pdf_path}: {e}')
    return text

def split_prepared_qa(text):
    """
    Split transcript into prepared remarks and Q&A section.
    Capital IQ transcripts typically have a clear Q&A separator.
    """
    # Common separators in Capital IQ transcripts
    qa_markers = [
        'Question-and-Answer Session',
        'QUESTION AND ANSWER',
        'Q&A Session',
        'Questions and Answers',
        'QUESTIONS AND ANSWERS',
    ]
    
    for marker in qa_markers:
        if marker.lower() in text.lower():
            idx = text.lower().find(marker.lower())
            return text[:idx], text[idx:]
    
    # If no marker found, return full text as prepared remarks
    return text, ''

# ── Load all transcripts ───────────────────────────────────────────────────────
transcripts = {}
for filename, quarter in transcript_map.items():
    filepath = os.path.join(TRANSCRIPT_DIR, filename)
    if os.path.exists(filepath):
        full_text = extract_transcript_text(filepath)
        prepared, qa = split_prepared_qa(full_text)
        transcripts[quarter] = {
            'full_text': full_text,
            'prepared': prepared,
            'qa': qa,
            'word_count': len(full_text.split()),
            'has_qa': len(qa) > 100
        }
        print(f'{quarter}: {len(full_text.split()):,} words, Q&A: {"Yes" if len(qa) > 100 else "No"}')
    else:
        print(f'{quarter}: FILE NOT FOUND — {filename}')

print(f'\nLoaded {len(transcripts)} transcripts')

Q2 2023: 6,912 words, Q&A: Yes
Q3 2023: 8,172 words, Q&A: Yes
Q4 2023: 8,314 words, Q&A: Yes
Q1 2024: 8,974 words, Q&A: Yes
Q2 2024: 10,369 words, Q&A: Yes
Q3 2024: 6,886 words, Q&A: Yes
Q4 2024: 9,517 words, Q&A: Yes
Q1 2025: 7,803 words, Q&A: Yes
Q2 2025: 10,446 words, Q&A: Yes
Q3 2025: 10,641 words, Q&A: Yes
Q4 2025: 10,611 words, Q&A: Yes

Loaded 11 transcripts


## Part 2: Sentiment Analysis — VADER vs FinBERT

I apply two complementary sentiment models at three levels:
1. **Full transcript** — overall tone
2. **Prepared remarks only** — management's scripted tone  
3. **Q&A only** — management's unscripted, spontaneous tone (often more revealing)

### Why Two Models?

**VADER** (Valence Aware Dictionary and sEntiment Reasoner)
- Rule-based, uses a predefined sentiment lexicon
- Fast, no training required
- Weakness: struggles with financial jargon and implicit negative language
  e.g. "navigating a fog" scores near-neutral in VADER but is clearly bearish

**FinBERT** (Financial BERT)
- Deep learning model fine-tuned on financial text (earnings calls, analyst 
  reports, financial news)
- Understands context and domain-specific language
- e.g. "miss", "headwind", "cautious outlook" are correctly scored as negative
- Weakness: slower, 512-token limit requires chunking long transcripts

**Our approach:** Use FinBERT as the primary scorer for financial 
interpretation, VADER as a baseline cross-check. When both models agree 
on direction, conviction is higher. Divergences reveal cases where 
financial context matters — exactly where FinBERT adds value over VADER.

In [3]:
# ── Install if needed ──────────────────────────────────────────────────────────
# !pip install transformers torch

from transformers import pipeline
import torch

# ── Load FinBERT ───────────────────────────────────────────────────────────────
print('Loading FinBERT...')
finbert = pipeline(
    'sentiment-analysis',
    model='ProsusAI/finbert',
    tokenizer='ProsusAI/finbert',
    device=0 if torch.cuda.is_available() else -1  # GPU if available, else CPU
)
print(f'FinBERT loaded. Device: {"GPU" if torch.cuda.is_available() else "CPU"}')

# ── FinBERT scoring function ───────────────────────────────────────────────────
def score_text_finbert(text, max_chunk_words=400):
    """
    Score text using FinBERT.
    FinBERT has a 512 token limit, so we chunk long texts.
    Returns compound score: positive=+1, negative=-1, neutral=0, weighted by confidence
    """
    if not text or len(text.strip()) < 10:
        return {'compound': 0, 'label': 'neutral', 'confidence': 0}

    # Split into sentences
    sentences = re.split(r'(?<=[.!?])\s+', text)
    sentences = [s.strip() for s in sentences if len(s.strip()) > 20]

    if not sentences:
        return {'compound': 0, 'label': 'neutral', 'confidence': 0}

    # FinBERT has 512 token limit — chunk sentences
    chunks = []
    current_chunk = []
    current_words = 0

    for sent in sentences:
        words = len(sent.split())
        if current_words + words > max_chunk_words and current_chunk:
            chunks.append(' '.join(current_chunk))
            current_chunk = [sent]
            current_words = words
        else:
            current_chunk.append(sent)
            current_words += words

    if current_chunk:
        chunks.append(' '.join(current_chunk))

    # Score each chunk
    scores = []
    for chunk in chunks:
        try:
            result = finbert(chunk[:512])[0]  # truncate to 512 chars as safety
            label = result['label'].lower()
            confidence = result['score']
            # Convert to compound: positive=+1, negative=-1, neutral=0
            if label == 'positive':
                compound = confidence
            elif label == 'negative':
                compound = -confidence
            else:
                compound = 0
            scores.append(compound)
        except Exception:
            scores.append(0)

    avg_compound = np.mean(scores) if scores else 0

    return {
        'compound': avg_compound,
        'label': 'positive' if avg_compound > 0.05 else 'negative' if avg_compound < -0.05 else 'neutral',
        'confidence': abs(avg_compound)
    }

# ── VADER scoring function (unchanged) ────────────────────────────────────────
def score_text_vader(text):
    """Score long text with VADER by averaging sentence-level scores"""
    if not text or len(text.strip()) < 10:
        return {'compound': 0, 'pos': 0, 'neg': 0, 'neu': 1}

    sentences = re.split(r'(?<=[.!?])\s+', text)
    sentences = [s.strip() for s in sentences if len(s.strip()) > 10]

    if not sentences:
        return {'compound': 0, 'pos': 0, 'neg': 0, 'neu': 1}

    scores = [analyzer.polarity_scores(s) for s in sentences]
    return {
        'compound': np.mean([s['compound'] for s in scores]),
        'pos': np.mean([s['pos'] for s in scores]),
        'neg': np.mean([s['neg'] for s in scores]),
        'neu': np.mean([s['neu'] for s in scores])
    }

# ── Score all transcripts with both models ─────────────────────────────────────
print('\nScoring transcripts with VADER and FinBERT...')
print('(FinBERT may take a few minutes on CPU)')

sentiment_results = []
for quarter, data in transcripts.items():
    print(f'  Processing {quarter}...', end=' ')

    # VADER scores
    vader_full = score_text_vader(data['full_text'])
    vader_prepared = score_text_vader(data['prepared'])
    vader_qa = score_text_vader(data['qa'])

    # FinBERT scores
    finbert_full = score_text_finbert(data['full_text'])
    finbert_prepared = score_text_finbert(data['prepared'])
    finbert_qa = score_text_finbert(data['qa'])

    sentiment_results.append({
        'quarter': quarter,
        # VADER
        'vader_full': vader_full['compound'],
        'vader_prepared': vader_prepared['compound'],
        'vader_qa': vader_qa['compound'],
        'vader_pos': vader_full['pos'],
        'vader_neg': vader_full['neg'],
        # FinBERT
        'finbert_full': finbert_full['compound'],
        'finbert_prepared': finbert_prepared['compound'],
        'finbert_qa': finbert_qa['compound'],
        # Meta
        'word_count': data['word_count'],
        'has_qa': data['has_qa']
    })
    print(f'VADER={vader_prepared["compound"]:.3f}, FinBERT={finbert_prepared["compound"]:.3f}')

sentiment_df = pd.DataFrame(sentiment_results)
quarter_order = list(transcript_map.values())
sentiment_df['quarter_idx'] = sentiment_df['quarter'].map(
    {q: i for i, q in enumerate(quarter_order)}
)
sentiment_df = sentiment_df.sort_values('quarter_idx').reset_index(drop=True)

print('\nSentiment comparison (VADER vs FinBERT):')
print(sentiment_df[['quarter', 'vader_prepared', 'finbert_prepared',
                     'vader_qa', 'finbert_qa']].round(3).to_string(index=False))

Loading FinBERT...


config.json:   0%|          | 0.00/758 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/252 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

FinBERT loaded. Device: CPU

Scoring transcripts with VADER and FinBERT...
(FinBERT may take a few minutes on CPU)
  Processing Q2 2023... VADER=0.224, FinBERT=0.000
  Processing Q3 2023... VADER=0.224, FinBERT=0.000
  Processing Q4 2023... VADER=0.232, FinBERT=0.000
  Processing Q1 2024... VADER=0.224, FinBERT=0.000
  Processing Q2 2024... VADER=0.224, FinBERT=0.000
  Processing Q3 2024... VADER=0.224, FinBERT=0.000
  Processing Q4 2024... VADER=0.232, FinBERT=0.000
  Processing Q1 2025... VADER=0.224, FinBERT=0.000
  Processing Q2 2025... VADER=0.222, FinBERT=0.000
  Processing Q3 2025... VADER=0.216, FinBERT=0.000
  Processing Q4 2025... VADER=0.231, FinBERT=0.000

Sentiment comparison (VADER vs FinBERT):
quarter  vader_prepared  finbert_prepared  vader_qa  finbert_qa
Q2 2023           0.224               0.0     0.288       0.295
Q3 2023           0.224               0.0     0.299       0.221
Q4 2023           0.232               0.0     0.301       0.340
Q1 2024           0.224   

---
## Part 3: Keyword Frequency Analysis

We track four keyword categories that capture key investment themes:
- **Growth signals** — bullish language about demand and momentum
- **Caution signals** — risk language about macro and competition  
- **SSSG-specific** — references to comparable store performance
- **Digital** — references to digital ordering and technology

In [ ]:
# ── Keyword categories ─────────────────────────────────────────────────────────
keyword_categories = {
    'growth_signals': [
        'traffic', 'momentum', 'strong', 'exceed', 'accelerat',
        'outperform', 'robust', 'healthy', 'positive', 'growth',
        'increase', 'expand', 'record', 'best'
    ],
    'caution_signals': [
        'uncertain', 'pressure', 'challeng', 'macro', 'cautious',
        'headwind', 'soften', 'decelerat', 'slow', 'difficult',
        'concern', 'risk', 'fog', 'navigate', 'volatile'
    ],
    'sssg_specific': [
        'same-restaurant', 'same restaurant', 'comparable',
        'comp sales', 'comps', 'same-store'
    ],
    'digital': [
        'digital', 'online', 'app', 'delivery', 'mobile',
        'technology', 'platform'
    ],
    'value_pricing': [
        'value', 'price', 'affordable', 'premium', 'ticket',
        'average check', 'menu price', 'inflation'
    ]
}

def count_keywords(text, keywords):
    """Count keyword occurrences per 1000 words (normalized)"""
    if not text:
        return 0
    text_lower = text.lower()
    word_count = len(text.split())
    count = sum(text_lower.count(kw.lower()) for kw in keywords)
    return (count / word_count) * 1000  # per 1000 words

# Count keywords for each transcript
keyword_results = []
for quarter, data in transcripts.items():
    row = {'quarter': quarter}
    for category, keywords in keyword_categories.items():
        # Score on full text and prepared remarks separately
        row[f'{category}_full'] = count_keywords(data['full_text'], keywords)
        row[f'{category}_prepared'] = count_keywords(data['prepared'], keywords)
    keyword_results.append(row)

keyword_df = pd.DataFrame(keyword_results)
keyword_df['quarter_idx'] = keyword_df['quarter'].map(
    {q: i for i, q in enumerate(quarter_order)}
)
keyword_df = keyword_df.sort_values('quarter_idx').reset_index(drop=True)

print('Keyword frequency (per 1000 words):')
display_cols = ['quarter'] + [f'{cat}_full' for cat in keyword_categories.keys()]
print(keyword_df[display_cols].round(2).to_string(index=False))

---
## Part 4: Merge with SSSG Data & Visualize

In [ ]:
# ── Load master data and merge ─────────────────────────────────────────────────
master = pd.read_csv(f'{BASE_DIR}/data/processed/cava_master_quarterly.csv',
                     parse_dates=['period_end'])

# Merge sentiment and keyword data
nlp_df = sentiment_df.merge(keyword_df, on=['quarter', 'quarter_idx'], how='outer')
nlp_df = nlp_df.merge(
    master[['quarter', 'sssg_pct', 'sssg_pct_next', 'period_end']],
    on='quarter', how='left'
)
nlp_df = nlp_df.sort_values('quarter_idx').reset_index(drop=True)

print(f'NLP dataset: {nlp_df.shape}')
print(nlp_df[['quarter', 'sentiment_full', 'sentiment_prepared', 
              'sentiment_qa', 'sssg_pct']].to_string(index=False))

In [ ]:
# ── Plot 1: Sentiment over time vs SSSG ───────────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(14, 10), sharex=True)
fig.suptitle('CAVA Earnings Call NLP Analysis', fontweight='bold', fontsize=14)

x = range(len(nlp_df))
quarters = nlp_df['quarter'].tolist()

# Top: SSSG vs Sentiment
ax1 = axes[0]
ax1_twin = ax1.twinx()

bars = ax1.bar(x, nlp_df['sssg_pct'],
               color=['#2ecc71' if v > 10 else '#e67e22' if v > 5 else '#e74c3c'
                      for v in nlp_df['sssg_pct'].fillna(0)],
               alpha=0.6, label='SSSG %')

ax1_twin.plot(x, nlp_df['sentiment_prepared'], marker='o', color='#3498db',
              linewidth=2, label='Prepared Remarks Sentiment')
ax1_twin.plot(x, nlp_df['sentiment_qa'], marker='s', color='#9b59b6',
              linewidth=2, linestyle='--', label='Q&A Sentiment')
ax1_twin.axhline(0, color='gray', linewidth=0.8, linestyle=':')

ax1.set_ylabel('SSSG (%)')
ax1_twin.set_ylabel('Sentiment Score')
ax1.set_title('SSSG vs Management Tone (Prepared Remarks vs Q&A)')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax1_twin.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2,
           loc='upper center', bbox_to_anchor=(0.5, -0.05), ncol=3)

# Bottom: Growth vs Caution keyword ratio
ax2 = axes[1]
ax2_twin = ax2.twinx()

ax2.plot(x, nlp_df['growth_signals_full'], marker='o', color='#2ecc71',
         linewidth=2, label='Growth keywords (per 1000w)')
ax2.plot(x, nlp_df['caution_signals_full'], marker='s', color='#e74c3c',
         linewidth=2, label='Caution keywords (per 1000w)')
ax2.set_ylabel('Keyword frequency (per 1000 words)')
ax2.set_title('Growth vs Caution Keyword Frequency')

# Growth/Caution ratio
nlp_df['gc_ratio'] = nlp_df['growth_signals_full'] / (
    nlp_df['caution_signals_full'] + 0.001)
ax2_twin.plot(x, nlp_df['gc_ratio'], marker='^', color='#e67e22',
              linewidth=2, linestyle=':', label='Growth/Caution ratio')
ax2_twin.axhline(1, color='gray', linewidth=0.8, linestyle=':')
ax2_twin.set_ylabel('Growth/Caution Ratio')

ax2.set_xticks(x)
ax2.set_xticklabels(quarters, rotation=45, ha='right')

lines3, labels3 = ax2.get_legend_handles_labels()
lines4, labels4 = ax2_twin.get_legend_handles_labels()
ax2.legend(lines3 + lines4, labels3 + labels4,
           loc='upper center', bbox_to_anchor=(0.5, -0.2), ncol=3)

plt.tight_layout()
plt.savefig(f'{BASE_DIR}/outputs/figures/earnings_nlp_sentiment.png',
            dpi=150, bbox_inches='tight')
plt.show()
print('Saved: outputs/figures/earnings_nlp_sentiment.png')

In [ ]:
# ── Plot 2: All keyword categories heatmap ─────────────────────────────────────
heatmap_cols = [f'{cat}_full' for cat in keyword_categories.keys()]
heatmap_df = nlp_df[['quarter'] + heatmap_cols].set_index('quarter')
heatmap_df.columns = list(keyword_categories.keys())

fig, ax = plt.subplots(figsize=(12, 6))
sns.heatmap(heatmap_df.T, annot=True, fmt='.1f', cmap='RdYlGn',
            ax=ax, linewidths=0.5, cbar_kws={'label': 'Frequency per 1000 words'})
ax.set_title('Keyword Frequency Heatmap by Quarter\n(green=high, red=low)',
             fontweight='bold')
ax.set_xlabel('')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig(f'{BASE_DIR}/outputs/figures/earnings_keyword_heatmap.png',
            dpi=150, bbox_inches='tight')
plt.show()
print('Saved: outputs/figures/earnings_keyword_heatmap.png')

In [ ]:
# ── Plot 3: Prepared vs Q&A sentiment gap ─────────────────────────────────────
# The gap between prepared and Q&A sentiment is revealing:
# A large negative gap (Q&A much more negative) suggests
# management is more cautious when pressed by analysts

nlp_df['sentiment_gap'] = nlp_df['sentiment_qa'] - nlp_df['sentiment_prepared']

fig, ax = plt.subplots(figsize=(13, 5))
colors = ['#e74c3c' if v < 0 else '#2ecc71' for v in nlp_df['sentiment_gap'].fillna(0)]
ax.bar(x, nlp_df['sentiment_gap'], color=colors, edgecolor='white')
ax.axhline(0, color='black', linewidth=0.8)
ax.set_xticks(x)
ax.set_xticklabels(quarters, rotation=45, ha='right')
ax.set_ylabel('Sentiment Gap (Q&A minus Prepared)')
ax.set_title('Q&A vs Prepared Remarks Sentiment Gap\n'
             '(negative = analysts drew out more cautious tone than prepared script)',
             fontweight='bold')
plt.tight_layout()
plt.savefig(f'{BASE_DIR}/outputs/figures/earnings_sentiment_gap.png',
            dpi=150, bbox_inches='tight')
plt.show()
print('Saved: outputs/figures/earnings_sentiment_gap.png')

---
## Part 5: Lead-Lag Analysis

Do NLP signals from earnings calls predict *next* quarter's SSSG?

In [ ]:
# ── Correlation: NLP signals vs next quarter SSSG ─────────────────────────────
nlp_corr_cols = [
    'sentiment_full', 'sentiment_prepared', 'sentiment_qa',
    'growth_signals_full', 'caution_signals_full',
    'gc_ratio', 'digital_full', 'value_pricing_full'
]

# Only use cols that exist
nlp_corr_cols = [c for c in nlp_corr_cols if c in nlp_df.columns]

print('Correlation with CURRENT quarter SSSG:')
corr_current = nlp_df[nlp_corr_cols + ['sssg_pct']].corr()['sssg_pct'].drop('sssg_pct')
print(corr_current.sort_values(ascending=False).round(3).to_string())

print('\nCorrelation with NEXT quarter SSSG (leading indicator test):')
corr_next = nlp_df[nlp_corr_cols + ['sssg_pct_next']].corr()['sssg_pct_next'].drop('sssg_pct_next')
print(corr_next.sort_values(ascending=False).round(3).to_string())

In [ ]:
# ── Visualization: NLP leading indicators ─────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('NLP Signal Correlations with SSSG', fontweight='bold')

# Current quarter
ax = axes[0]
corr_current_sorted = corr_current.sort_values()
colors = ['#2ecc71' if v > 0 else '#e74c3c' for v in corr_current_sorted]
ax.barh(corr_current_sorted.index, corr_current_sorted.values,
        color=colors, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('vs Current Quarter SSSG')
ax.set_xlabel('Correlation')

# Next quarter
ax = axes[1]
corr_next_sorted = corr_next.sort_values()
colors = ['#2ecc71' if v > 0 else '#e74c3c' for v in corr_next_sorted]
ax.barh(corr_next_sorted.index, corr_next_sorted.values,
        color=colors, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('vs NEXT Quarter SSSG (Leading Indicator)')
ax.set_xlabel('Correlation')

plt.tight_layout()
plt.savefig(f'{BASE_DIR}/outputs/figures/nlp_correlations.png',
            dpi=150, bbox_inches='tight')
plt.show()
print('Saved: outputs/figures/nlp_correlations.png')

---
## Part 6: Key Quote Extraction

Extract the most sentiment-charged sentences from each transcript
to provide qualitative evidence for our investment thesis.

In [ ]:
# ── Extract most negative and positive sentences ───────────────────────────────
def extract_key_sentences(text, n=3, sentiment='negative'):
    """Extract most positive or negative sentences from transcript"""
    sentences = re.split(r'(?<=[.!?])\s+', text)
    sentences = [s.strip() for s in sentences
                 if len(s.strip()) > 30 and len(s.strip()) < 300]

    scored = [(s, analyzer.polarity_scores(s)['compound']) for s in sentences]

    if sentiment == 'negative':
        scored.sort(key=lambda x: x[1])
    else:
        scored.sort(key=lambda x: x[1], reverse=True)

    return scored[:n]

# Focus on key quarters: Q1 2024 miss, Q2 2025 miss, Q4 2024 beat
key_quarters = ['Q2 2025', 'Q3 2025', 'Q4 2024']

for quarter in key_quarters:
    if quarter in transcripts:
        print(f'\n=== {quarter} — Most Negative Sentences ===')
        neg_sentences = extract_key_sentences(
            transcripts[quarter]['prepared'], n=3, sentiment='negative'
        )
        for sent, score in neg_sentences:
            print(f'  [{score:.3f}] {sent[:150]}...')

        print(f'\n=== {quarter} — Most Positive Sentences ===')
        pos_sentences = extract_key_sentences(
            transcripts[quarter]['prepared'], n=3, sentiment='positive'
        )
        for sent, score in pos_sentences:
            print(f'  [{score:.3f}] {sent[:150]}...')

---
## Part 7: NLP Findings & Investment Interpretation

In [ ]:
# ── Summary statistics ─────────────────────────────────────────────────────────
print('=== NLP ANALYSIS SUMMARY ===')
print(f'Transcripts analyzed: {len(transcripts)}')
print(f'Coverage: Q2 2023 – Q4 2025')
print(f'\nSentiment trend (prepared remarks):')
print(f'  Early period (Q2-Q4 2023): {nlp_df[nlp_df["quarter_idx"] <= 2]["sentiment_prepared"].mean():.3f}')
print(f'  Growth peak (Q1-Q4 2024):  {nlp_df[(nlp_df["quarter_idx"] >= 3) & (nlp_df["quarter_idx"] <= 6)]["sentiment_prepared"].mean():.3f}')
print(f'  Decel period (2025):       {nlp_df[nlp_df["quarter_idx"] >= 7]["sentiment_prepared"].mean():.3f}')

print(f'\nGrowth/Caution ratio trend:')
print(nlp_df[['quarter', 'gc_ratio']].round(2).to_string(index=False))

print(f'\nBest NLP leading indicator of next-quarter SSSG:')
print(corr_next.abs().sort_values(ascending=False).head(3).round(3).to_string())

In [ ]:
# ── Save NLP results ───────────────────────────────────────────────────────────
nlp_df.to_csv(f'{BASE_DIR}/data/processed/cava_earnings_nlp.csv', index=False)
print('Saved: data/processed/cava_earnings_nlp.csv')

print('\nFigures saved:')
print('  outputs/figures/earnings_nlp_sentiment.png')
print('  outputs/figures/earnings_keyword_heatmap.png')
print('  outputs/figures/earnings_sentiment_gap.png')
print('  outputs/figures/nlp_correlations.png')